# Polynomial Interpolation with Optimizer Benchmarking

This notebook generalizes the polynomial learning problem to **polynomial interpolation**.

## Problem Statement

Given data points $(x_i, y_i)$ where $x_i, y_i$ are p-adic numbers, learn polynomial coefficients $(a_0, \ldots, a_n)$ such that:

$$f(x_i) = a_0 + a_1 x_i + \cdots + a_n x_i^n \approx y_i$$

This is a generalization of the root-finding problem where $y_i = 0$ for all $i$.

## Optimizer Benchmarking

We compare the following optimizers:
1. **Greedy Descent** (classical)
2. **MCTS** (Monte Carlo Tree Search)
3. **UCT** (Upper Confidence Trees)
4. **Modified UCT** (variant)
5. **Flat UCB** (flat variant)
6. **HOO** (Hierarchical Optimistic Optimization)

## 1. Setup and Imports

In [1]:
println("Loading libraries...")
include("../../src/NAML.jl")
include("util.jl")

using Oscar
using .NAML
using Printf

println("Libraries loaded successfully!")

Loading libraries...
  ___   ___   ___    _    ____
 / _ \ / __\ / __\  / \  |  _ \  | Combining and extending ANTIC, GAP,
| |_| |\__ \| |__  / ^ \ |  ´ /  | Polymake and Singular
 \___/ \___/ \___//_/ \_\|_|\_\  | Type "?Oscar" for more information
o--------o-----o-----o--------o  | Documentation: https://docs.oscar-system.org
  S Y M B O L I C   T O O L S    | Version 1.6.0
Libraries loaded successfully!


## 2. Configuration

In [2]:
# Problem configuration
prec = 20                # p-adic precision
p = 2                    # prime
K = PadicField(p, prec)  # p-adic field

n_points = 5             # number of data points
poly_degree = 4          # polynomial degree (we have degree+1 parameters)
n_epochs = 15            # number of optimization epochs

println("="^70)
println("Polynomial Interpolation Configuration")
println("="^70)
println("Prime:             $p")
println("Precision:         $prec")
println("Data points:       $n_points")
println("Polynomial degree: $poly_degree")
println("Parameters:        $(poly_degree + 1) coefficients")
println("Epochs:            $n_epochs")
println("="^70)

Polynomial Interpolation Configuration
Prime:             2
Precision:         20
Data points:       5
Polynomial degree: 4
Parameters:        5 coefficients
Epochs:            15


## 3. Generate Interpolation Data

We generate random p-adic data points $(x_i, y_i)$ where **both** $x_i$ and $y_i$ are p-adic numbers.

This is different from the original notebook which used $y_i = 0$ (root finding).

In [3]:
println("\nGenerating interpolation data...")

# Generate random x values (distinct)
x_values = Vector{PadicFieldElem}()
max_attempts = n_points * 100
attempts = 0

while length(x_values) < n_points && attempts < max_attempts
    x = generate_random_padic(p, prec, 0, 8)  # integers with 8 terms
    if !any(existing_x -> existing_x == x, x_values)
        push!(x_values, x)
    end
    attempts += 1
end

# Generate random y values (p-adic, not necessarily zero!)
y_values = [generate_random_padic(p, prec, 0, 8) for _ in 1:n_points]

# Create data pairs
data = [(x, y) for (x, y) in zip(x_values, y_values)]

println("\nData points (x_i, y_i):")
for (i, (x, y)) in enumerate(data)
    println("  Point $i: x = $x, y = $y")
end

# Verify distinctness
all_distinct = length(x_values) == length(unique(x_values))
println("\nAll x values distinct: $all_distinct")


Generating interpolation data...

Data points (x_i, y_i):
  Point 1: x = 2^3 + 2^4 + 2^5 + 2^6 + 2^7 + O(2^20), y = 2^0 + 2^2 + 2^6 + O(2^20)
  Point 2: x = 2^3 + 2^4 + 2^6 + O(2^20), y = 2^1 + 2^4 + 2^6 + 2^7 + O(2^20)
  Point 3: x = 2^2 + 2^3 + O(2^20), y = 2^0 + 2^2 + 2^7 + O(2^20)
  Point 4: x = 2^0 + 2^4 + 2^5 + 2^6 + O(2^20), y = 2^1 + 2^2 + 2^3 + 2^4 + 2^5 + 2^6 + O(2^20)
  Point 5: x = 2^0 + 2^2 + 2^3 + 2^7 + O(2^20), y = 2^1 + 2^5 + 2^7 + O(2^20)

All x values distinct: true


## 4. Create Loss Function

Since both $x_i$ and $y_i$ are p-adic, we use the p-adic case of `polynomial_to_linear_loss`.

The loss is: $$L(a_0, \ldots, a_n) = \sum_{i=1}^{n} |a_0 + a_1 x_i + \cdots + a_n x_i^n - y_i|^2$$

In [4]:
println("\nCreating loss function...")

# Use polynomial_to_linear_loss with p-adic outputs (no cutoff needed)
loss = polynomial_to_linear_loss(data, poly_degree, nothing)

println("Loss function created successfully")
println("Loss type: Mean squared error (p-adic)")
println("Parameters: $(poly_degree + 1) coefficients (a₀, ..., a$poly_degree)")


Creating loss function...
Loss function created successfully
Loss type: Mean squared error (p-adic)
Parameters: 5 coefficients (a₀, ..., a4)


## 5. Initialize Starting Point

All optimizers start from the same Gauss point for fair comparison.

In [5]:
println("\nInitializing starting point...")

# Start at Gauss point (standard starting point)
initial_param = generate_gauss_point(poly_degree + 1, K)

println("Starting point: Gauss point")
println("Center: $(NAML.center(initial_param))")
println("Radius: $(NAML.radius(initial_param))")

# Compute initial loss
initial_loss = loss.eval([initial_param])[1]
println("\nInitial loss: $initial_loss")


Initializing starting point...
Starting point: Gauss point
Center: PadicFieldElem[2^0 + O(2^20), 2^0 + O(2^20), 2^0 + O(2^20), 2^0 + O(2^20), 2^0 + O(2^20)]
Radius: [0, 0, 0, 0, 0]

Initial loss: 5.0


## 6. Benchmark Optimizers

We now run all optimizers and collect:
- Final loss value
- Total time taken
- Loss trajectory over epochs

### 6.1 Greedy Descent

In [6]:
println("\n" * "="^70)
println("Optimizer 1: Greedy Descent")
println("="^70)

# Initialize optimizer
state = 1
config = (true, 0)  # Force all coordinates to shrink progressively
optim_greedy = NAML.greedy_descent_init(initial_param, loss, state, config)

# Track loss trajectory
greedy_losses = Float64[]

# Run optimization
t_start = time()
for epoch in 1:n_epochs
    current_loss = NAML.eval_loss(optim_greedy)
    push!(greedy_losses, current_loss)
    println(@sprintf("  Epoch %2d: loss = %.6e", epoch, current_loss))
    NAML.step!(optim_greedy)
end
t_end = time()
greedy_time = t_end - t_start

# Final results
greedy_final_loss = NAML.eval_loss(optim_greedy)
push!(greedy_losses, greedy_final_loss)

println("\nGreedy Descent Summary:")
println(@sprintf("  Time: %.3f seconds", greedy_time))
println(@sprintf("  Final loss: %.6e", greedy_final_loss))
println(@sprintf("  Improvement: %.6e", initial_loss - greedy_final_loss))


Optimizer 1: Greedy Descent
  Epoch  1: loss = 5.000000e+00
  Epoch  2: loss = 3.500000e+00
  Epoch  3: loss = 3.500000e+00
  Epoch  4: loss = 3.500000e+00
  Epoch  5: loss = 3.500000e+00
  Epoch  6: loss = 2.000000e+00
  Epoch  7: loss = 2.000000e+00
  Epoch  8: loss = 2.000000e+00
  Epoch  9: loss = 2.000000e+00
  Epoch 10: loss = 2.000000e+00
  Epoch 11: loss = 2.000000e+00
  Epoch 12: loss = 2.000000e+00
  Epoch 13: loss = 2.000000e+00
  Epoch 14: loss = 2.000000e+00
  Epoch 15: loss = 2.000000e+00

Greedy Descent Summary:
  Time: 0.450 seconds
  Final loss: 2.000000e+00
  Improvement: 3.000000e+00


### 6.2 MCTS (Monte Carlo Tree Search)

In [8]:
println("\n" * "="^70)
println("Optimizer 2: MCTS")
println("="^70)

# Initialize optimizer
mcts_config = NAML.MCTSConfig(
    num_simulations=100,
    exploration_constant=1.41,
    selection_mode=NAML.BestValue,
    degree=3
)
optim_mcts = NAML.mcts_descent_init(initial_param, loss, mcts_config)

# Track loss trajectory
mcts_losses = Float64[]

# Run optimization
t_start = time()
for epoch in 1:n_epochs
    current_loss = NAML.eval_loss(optim_mcts)
    push!(mcts_losses, current_loss)
    println(@sprintf("  Epoch %2d: loss = %.6e", epoch, current_loss))
    NAML.step!(optim_mcts)
end
t_end = time()
mcts_time = t_end - t_start

# Final results
mcts_final_loss = NAML.eval_loss(optim_mcts)
push!(mcts_losses, mcts_final_loss)

println("\nMCTS Summary:")
println(@sprintf("  Time: %.3f seconds", mcts_time))
println(@sprintf("  Final loss: %.6e", mcts_final_loss))
println(@sprintf("  Improvement: %.6e", initial_loss - mcts_final_loss))


Optimizer 2: MCTS
  Epoch  1: loss = 5.000000e+00
  Epoch  2: loss = 3.500000e+00
  Epoch  3: loss = 3.500000e+00
  Epoch  4: loss = 1.625000e+00
  Epoch  5: loss = 1.203125e+00
  Epoch  6: loss = 1.191406e+00
  Epoch  7: loss = 1.141602e+00
  Epoch  8: loss = 1.140869e+00
  Epoch  9: loss = 1.128967e+00
  Epoch 10: loss = 1.126038e+00
  Epoch 11: loss = 1.125992e+00
  Epoch 12: loss = 1.125259e+00
  Epoch 13: loss = 1.125248e+00
  Epoch 14: loss = 1.125062e+00
  Epoch 15: loss = 1.125016e+00

MCTS Summary:
  Time: 0.767 seconds
  Final loss: 1.125015e+00
  Improvement: 3.874985e+00


### 6.3 UCT (Upper Confidence Trees)

In [9]:
println("\n" * "="^70)
println("Optimizer 3: UCT")
println("="^70)

# Initialize optimizer
uct_config = NAML.UCTConfig(
    max_depth=10,
    num_simulations=100,
    exploration_constant=1.41,
    degree=3
)
uct_state = NAML.UCTState{PadicFieldElem, Int}(Dict(), 0)
optim_uct = NAML.uct_descent_init(initial_param, loss, uct_state, uct_config)

# Track loss trajectory
uct_losses = Float64[]

# Run optimization
t_start = time()
for epoch in 1:n_epochs
    current_loss = NAML.eval_loss(optim_uct)
    push!(uct_losses, current_loss)
    println(@sprintf("  Epoch %2d: loss = %.6e", epoch, current_loss))
    NAML.step!(optim_uct)
end
t_end = time()
uct_time = t_end - t_start

# Final results
uct_final_loss = NAML.eval_loss(optim_uct)
push!(uct_losses, uct_final_loss)

println("\nUCT Summary:")
println(@sprintf("  Time: %.3f seconds", uct_time))
println(@sprintf("  Final loss: %.6e", uct_final_loss))
println(@sprintf("  Improvement: %.6e", initial_loss - uct_final_loss))


Optimizer 3: UCT


LoadError: MethodError: [0mCannot `convert` an object of type [92mDict{Any, Any}[39m[0m to an object of type [91mUCTNode{PadicFieldElem, Int64}[39m
The function `convert` exists, but no method is defined for this combination of argument types.

[0mClosest candidates are:
[0m  (::Type{UCTNode{S, T}} where {S, T})(::Any, [91m::Any[39m, [91m::Any[39m, [91m::Any[39m, [91m::Any[39m, [91m::Any[39m)
[0m[90m   @[39m [35mMain.NAML[39m [90m~/Documents/naml/src/optimization/optimizers/tree_search/[39m[90m[4muct.jl:25[24m[39m
[0m  convert(::Type{T}, [91m::T[39m) where T
[0m[90m   @[39m [90mBase[39m [90m[4mBase_compiler.jl:133[24m[39m


### 6.4 Modified UCT

In [10]:
println("\n" * "="^70)
println("Optimizer 4: Modified UCT")
println("="^70)

# Initialize optimizer
mod_uct_config = NAML.ModifiedUCTConfig(
    max_depth=10,
    num_simulations=100,
    exploration_constant=1.41,
    degree=3
)
mod_uct_state = NAML.ModifiedUCTState{PadicFieldElem, Int}(Dict(), 0)
optim_mod_uct = NAML.modified_uct_descent_init(initial_param, loss, mod_uct_state, mod_uct_config)

# Track loss trajectory
mod_uct_losses = Float64[]

# Run optimization
t_start = time()
for epoch in 1:n_epochs
    current_loss = NAML.eval_loss(optim_mod_uct)
    push!(mod_uct_losses, current_loss)
    println(@sprintf("  Epoch %2d: loss = %.6e", epoch, current_loss))
    NAML.step!(optim_mod_uct)
end
t_end = time()
mod_uct_time = t_end - t_start

# Final results
mod_uct_final_loss = NAML.eval_loss(optim_mod_uct)
push!(mod_uct_losses, mod_uct_final_loss)

println("\nModified UCT Summary:")
println(@sprintf("  Time: %.3f seconds", mod_uct_time))
println(@sprintf("  Final loss: %.6e", mod_uct_final_loss))
println(@sprintf("  Improvement: %.6e", initial_loss - mod_uct_final_loss))


Optimizer 4: Modified UCT


LoadError: MethodError: no method matching ModifiedUCTConfig(; max_depth::Int64, num_simulations::Int64, exploration_constant::Float64, degree::Int64)
This method does not support all of the given keyword arguments (and may not support any).

[0mClosest candidates are:
[0m  ModifiedUCTConfig(; max_depth, num_simulations, beta, degree, strict, value_transform)[91m got unsupported keyword argument "exploration_constant"[39m
[0m[90m   @[39m [35mMain.NAML[39m [90m~/Documents/naml/src/optimization/optimizers/tree_search/[39m[90m[4mmodified_uct.jl:139[24m[39m
[0m  ModifiedUCTConfig([91m::Int64[39m, [91m::Int64[39m, [91m::Float64[39m, [91m::Int64[39m, [91m::Vector{Float64}[39m, [91m::Vector{Float64}[39m, [91m::Int64[39m, [91m::Bool[39m, [91m::Function[39m)[91m got unsupported keyword arguments "max_depth", "num_simulations", "exploration_constant", "degree"[39m
[0m[90m   @[39m [35mMain.NAML[39m [90m~/Documents/naml/src/optimization/optimizers/tree_sea

### 6.5 Flat UCB

In [11]:
println("\n" * "="^70)
println("Optimizer 5: Flat UCB")
println("="^70)

# Initialize optimizer
flat_ucb_config = NAML.FlatUCBConfig(
    max_iterations=100,
    exploration_constant=1.41,
    degree=3
)
flat_ucb_state = NAML.FlatUCBState{PadicFieldElem, Int}(Dict(), 0)
optim_flat_ucb = NAML.flat_ucb_descent_init(initial_param, loss, flat_ucb_state, flat_ucb_config)

# Track loss trajectory
flat_ucb_losses = Float64[]

# Run optimization
t_start = time()
for epoch in 1:n_epochs
    current_loss = NAML.eval_loss(optim_flat_ucb)
    push!(flat_ucb_losses, current_loss)
    println(@sprintf("  Epoch %2d: loss = %.6e", epoch, current_loss))
    NAML.step!(optim_flat_ucb)
end
t_end = time()
flat_ucb_time = t_end - t_start

# Final results
flat_ucb_final_loss = NAML.eval_loss(optim_flat_ucb)
push!(flat_ucb_losses, flat_ucb_final_loss)

println("\nFlat UCB Summary:")
println(@sprintf("  Time: %.3f seconds", flat_ucb_time))
println(@sprintf("  Final loss: %.6e", flat_ucb_final_loss))
println(@sprintf("  Improvement: %.6e", initial_loss - flat_ucb_final_loss))


Optimizer 5: Flat UCB


LoadError: MethodError: no method matching FlatUCBConfig(; max_iterations::Int64, exploration_constant::Float64, degree::Int64)
This method does not support all of the given keyword arguments (and may not support any).

[0mClosest candidates are:
[0m  FlatUCBConfig(; max_depth, num_simulations, beta, degree, strict, value_transform)[91m got unsupported keyword arguments "max_iterations", "exploration_constant"[39m
[0m[90m   @[39m [35mMain.NAML[39m [90m~/Documents/naml/src/optimization/optimizers/tree_search/[39m[90m[4mflat_ucb.jl:143[24m[39m
[0m  FlatUCBConfig([91m::Int64[39m, [91m::Int64[39m, [91m::Float64[39m, [91m::Int64[39m, [91m::Int64[39m, [91m::Bool[39m, [91m::Function[39m)[91m got unsupported keyword arguments "max_iterations", "exploration_constant", "degree"[39m
[0m[90m   @[39m [35mMain.NAML[39m [90m~/Documents/naml/src/optimization/optimizers/tree_search/[39m[90m[4mflat_ucb.jl:102[24m[39m
[0m  FlatUCBConfig([91m::Any[39m, [91m::

### 6.6 HOO (Hierarchical Optimistic Optimization)

In [12]:
println("\n" * "="^70)
println("Optimizer 6: HOO")
println("="^70)

# Initialize optimizer
hoo_config = NAML.HOOConfig(
    rho=0.5,
    nu1=0.1,
    max_depth=15
)
optim_hoo = NAML.hoo_descent_init(initial_param, loss, 1, hoo_config)

# Track loss trajectory
hoo_losses = Float64[]

# Run optimization
t_start = time()
for epoch in 1:n_epochs
    current_loss = NAML.eval_loss(optim_hoo)
    push!(hoo_losses, current_loss)
    println(@sprintf("  Epoch %2d: loss = %.6e", epoch, current_loss))
    NAML.step!(optim_hoo)
end
t_end = time()
hoo_time = t_end - t_start

# Final results
hoo_final_loss = NAML.eval_loss(optim_hoo)
push!(hoo_losses, hoo_final_loss)

# Get tree statistics
tree_size = NAML.get_tree_size(optim_hoo.state)
visited = NAML.get_visited_nodes(optim_hoo.state)
leaves = NAML.get_leaf_nodes(optim_hoo.state)

println("\nHOO Summary:")
println(@sprintf("  Time: %.3f seconds", hoo_time))
println(@sprintf("  Final loss: %.6e", hoo_final_loss))
println(@sprintf("  Improvement: %.6e", initial_loss - hoo_final_loss))
println("  Tree size: $tree_size nodes")
println("  Visited nodes: $visited")
println("  Leaf nodes: $leaves")


Optimizer 6: HOO


LoadError: MethodError: no method matching hoo_descent_init(::ValuationPolydisc{PadicFieldElem, Int64}, ::Loss, ::Int64, ::HOOConfig)
The function `hoo_descent_init` exists, but no method is defined for this combination of argument types.

[0mClosest candidates are:
[0m  hoo_descent_init(::ValuationPolydisc{S, T}, ::Loss, [91m::HOOConfig[39m) where {S, T}
[0m[90m   @[39m [35mMain.NAML[39m [90m~/Documents/naml/src/optimization/optimizers/tree_search/[39m[90m[4mhoo.jl:511[24m[39m
[0m  hoo_descent_init(::ValuationPolydisc{S, T}, ::Loss) where {S, T}
[0m[90m   @[39m [35mMain.NAML[39m [90m~/Documents/naml/src/optimization/optimizers/tree_search/[39m[90m[4mhoo.jl:511[24m[39m


## 7. Comparison Table

In [13]:
println("\n" * "="^70)
println("Optimizer Comparison")
println("="^70)
println()
println(@sprintf("%-20s %15s %15s %15s", "Optimizer", "Time (s)", "Final Loss", "Improvement"))
println("-"^70)
println(@sprintf("%-20s %15.3f %15.6e %15.6e", "Greedy Descent", greedy_time, greedy_final_loss, initial_loss - greedy_final_loss))
println(@sprintf("%-20s %15.3f %15.6e %15.6e", "MCTS", mcts_time, mcts_final_loss, initial_loss - mcts_final_loss))
println(@sprintf("%-20s %15.3f %15.6e %15.6e", "UCT", uct_time, uct_final_loss, initial_loss - uct_final_loss))
println(@sprintf("%-20s %15.3f %15.6e %15.6e", "Modified UCT", mod_uct_time, mod_uct_final_loss, initial_loss - mod_uct_final_loss))
println(@sprintf("%-20s %15.3f %15.6e %15.6e", "Flat UCB", flat_ucb_time, flat_ucb_final_loss, initial_loss - flat_ucb_final_loss))
println(@sprintf("%-20s %15.3f %15.6e %15.6e", "HOO", hoo_time, hoo_final_loss, initial_loss - hoo_final_loss))
println("="^70)


Optimizer Comparison

Optimizer                   Time (s)      Final Loss     Improvement
----------------------------------------------------------------------
Greedy Descent                 0.450    2.000000e+00    3.000000e+00
MCTS                           0.767    1.125015e+00    3.874985e+00


LoadError: UndefVarError: `uct_time` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

## 8. Loss Trajectories

Display the loss evolution for each optimizer over epochs.

In [ ]:
println("\n" * "="^70)
println("Loss Trajectories")
println("="^70)
println()

# Find best and worst loss at each epoch
println(@sprintf("%-6s %15s %15s %15s %15s %15s %15s", "Epoch", "Greedy", "MCTS", "UCT", "Mod-UCT", "Flat-UCB", "HOO"))
println("-"^100)

for epoch in 1:length(greedy_losses)
    println(@sprintf("%-6d %15.6e %15.6e %15.6e %15.6e %15.6e %15.6e",
                    epoch - 1,
                    greedy_losses[epoch],
                    mcts_losses[epoch],
                    uct_losses[epoch],
                    mod_uct_losses[epoch],
                    flat_ucb_losses[epoch],
                    hoo_losses[epoch]))
end
println("="^100)

## 9. Analysis and Insights

### Key Observations:

1. **Greedy Descent**: Classical baseline, typically fast but may get stuck in local minima
2. **Tree Search Methods**: MCTS, UCT, Modified UCT, Flat UCB, HOO explore the parameter space more systematically
3. **Trade-offs**: Tree search methods may take longer per epoch but can find better solutions

### Understanding the Results:

- **Lower final loss** = better fit to the data
- **Faster time** = more efficient optimization
- **Loss trajectory** shows convergence behavior

### Problem Context:

We're learning polynomial coefficients $(a_0, \ldots, a_n)$ to interpolate p-adic data points. The optimization happens in p-adic space, which has a tree structure that tree search algorithms can exploit efficiently.

### Next Steps:

- Try different polynomial degrees
- Vary the number of data points
- Tune hyperparameters (exploration constants, simulation counts, etc.)
- Compare with gradient-based methods